### LEO-vetter Practice

In [3]:
import numpy as np
import lightkurve as lk
import pandas as pd

from astroquery.mast import Catalogs

from leo_vetter.stellar import quadratic_ldc
from leo_vetter.main import TCELightCurve
from leo_vetter.plots import plot_modshift, plot_summary
from leo_vetter.thresholds import check_thresholds

In [25]:
def mdwarf_targets(df, n_targets=5):
    """
    Choose 1-5 good planet targets from nearby_TOI_MDs.csv.

    Criteria:
    - M-dwarf host stars (checked with Teff < 4000 K)
    - short orbital period preferred
    - large transit depth preferred
    - large planet radius preferred

    Parameters
    ----------
    csv_file : str
        Path to the CSV file.
    n_targets : int
        Number of targets to return (1 to 5 recommended).

    Returns
    -------
    pandas.DataFrame
        Top-ranked targets.
    """
    # Keep only rows with needed values
    df = df.dropna(subset=[
        "Detecion Pipeline(s)",
        "Sectors"
        "Orbital Period (days) Value",
        "Transit Depth Value",
        "Planet Radius Value",
        "Effective Temperature Value"
    ]).copy()

    # Optional safety check for M-dwarfs
    df = df[df["Effective Temperature Value"].between(2000, 3500)].copy()

    # Optional: keep likely planet candidates / confirmed planets only
    # CP = Confirmed Planet, PC = Planet Candidate
    df = df[df["TOI Disposition"].isin(["CP", "PC"])].copy()

    # Build a score
    # shorter period is better -> use inverse period
    df["period_score"] = 1 / df["Orbital Period (days) Value"]

    # Normalize data for different units
    def normalize(series):
        smin = series.min()
        smax = series.max()
        if smax == smin:
            return pd.Series([1.0] * len(series), index=series.index)
        return (series - smin) / (smax - smin)

    df["period_score_norm"] = normalize(df["period_score"])
    df["depth_norm"] = normalize(df["Transit Depth Value"])
    df["radius_norm"] = normalize(df["Planet Radius Value"])

    # Combined score
    df["target_score"] = (
        0.45 * df["period_score_norm"] +
        0.35 * df["depth_norm"] +
        0.20 * df["radius_norm"]
    )

    # Sort from best to worst
    df = df.sort_values("target_score", ascending=False)

    # Return useful columns only
    return df.head(n_targets)[[
        "Full TOI ID",
        "TIC ID",
        "TOI Disposition",
        "TMag Value",
        "Effective Temperature Value",
        "Orbital Period (days) Value",
        "Transit Depth Value",
        "Planet Radius Value",
        "target_score"
    ]]

In [26]:
df = pd.read_csv('/Users/madelinejmg/Desktop/PHYS39002/research2026/Catalog/nearby_TOI_MDs.csv')

top_targets = mdwarf_targets(df, n_targets=5)
print(top_targets)

KeyError: ['Detecion Pipeline(s)', 'SectorsOrbital Period (days) Value']